In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")
RANDOM_STATE = 42

# Load
df = pd.read_csv("marketing_campaign.csv", sep="\t")

# Clean
df = df.dropna(subset=["Income"]).copy()
df["Age"] = 2026 - df["Year_Birth"]
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
df["Days_As_Customer"] = (df["Dt_Customer"].max() - df["Dt_Customer"]).dt.days
df = df[(df["Age"] < 90) & (df["Income"] < 200000)].reset_index(drop=True)

df["Education"] = df["Education"].replace({
    "Basic": "Undergraduate", "2n Cycle": "Undergraduate",
    "Graduation": "Graduate", "Master": "Postgraduate", "PhD": "Postgraduate"
})
df["Marital_Status"] = df["Marital_Status"].replace({
    "Married": "Partner", "Together": "Partner",
    "Single": "Alone", "Divorced": "Alone", "Widow": "Alone",
    "Alone": "Alone", "Absurd": "Alone", "YOLO": "Alone"
})

df["Total_Spend"] = df[["MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds"]].sum(axis=1)
df["Total_Purchases"] = df[["NumDealsPurchases","NumWebPurchases","NumCatalogPurchases","NumStorePurchases"]].sum(axis=1)
df["Total_Children"] = df["Kidhome"] + df["Teenhome"]

feature_cols = [
    "Education","Marital_Status","Income","Kidhome","Teenhome","Recency",
    "MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds",
    "NumDealsPurchases","NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumWebVisitsMonth",
    "AcceptedCmp1","AcceptedCmp2","AcceptedCmp3","AcceptedCmp4","AcceptedCmp5","Response","Complain",
    "Age","Days_As_Customer"
]

X_raw = df[feature_cols].copy()
X_encoded = pd.get_dummies(X_raw, columns=["Education","Marital_Status"], drop_first=True)

print("Total features before PCA:", X_encoded.shape[1])

# Phase 1: SCALE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

# Phase 2: COMPRESS (PCA)
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o")
plt.axhline(0.95, color="orange", linestyle="--", label="95% Threshold")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Cumulative Explained Variance")
plt.legend()
plt.show()

N_COMPONENTS = 3
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
pca_cols = [f"PC{i+1}" for i in range(N_COMPONENTS)]
df_pca = pd.DataFrame(X_pca, columns=pca_cols)

print("Explained variance by", N_COMPONENTS, "components:", pca.explained_variance_ratio_.sum().round(4))

# Phase 3: CLUSTER (K-Means) - Elbow Method + Silhouette Score
K_range = range(2, 11)
wcss = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_pca)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), wcss, marker="o")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("WCSS")
axes[0].set_title("Elbow Method")

axes[1].plot(list(K_range), sil_scores, marker="o", color="darkorange")
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score")
plt.tight_layout()
plt.show()

diffs = np.diff(wcss)
diffs2 = np.diff(diffs)
elbow_k = list(K_range)[int(np.argmax(diffs2)) + 1] if len(diffs2) > 0 else list(K_range)[0]
silhouette_k = list(K_range)[int(np.argmax(sil_scores))]

print("Elbow Method suggested K:", elbow_k)
print("Silhouette Score suggested K:", silhouette_k)

OPTIMAL_K = silhouette_k
print("Final optimal K used:", OPTIMAL_K)

kmeans_final = KMeans(n_clusters=OPTIMAL_K, init="k-means++", n_init=10, random_state=RANDOM_STATE)
cluster_labels = kmeans_final.fit_predict(X_pca)

df["Cluster"] = cluster_labels
df_pca["Cluster"] = cluster_labels

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(df_pca["PC1"], df_pca["PC2"], df_pca["PC3"], c=cluster_labels, cmap="viridis", s=30)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("Customer Segments in PCA Space (3D)")
legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend1)
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(x="PC1", y="PC2", hue="Cluster", data=df_pca, palette="viridis", s=50)
plt.title("Customer Segments in PCA Space (2D)")
plt.show()

# Phase 4: TRANSLATE (Business Personas)
persona_features = ["Age","Income","Total_Spend","Total_Purchases","Total_Children","Recency","NumWebVisitsMonth","Days_As_Customer"]

persona_summary = df.groupby("Cluster")[persona_features].mean().round(1)
persona_summary["Count"] = df["Cluster"].value_counts().sort_index()

income_median = persona_summary["Income"].median()
spend_median = persona_summary["Total_Spend"].median()

def assign_persona(row):
    if row["Income"] >= income_median and row["Total_Spend"] >= spend_median:
        return "High-Value Trendsetters"
    elif row["Income"] >= income_median and row["Total_Spend"] < spend_median:
        return "Affluent Conservatives"
    elif row["Income"] < income_median and row["Total_Spend"] >= spend_median:
        return "Budget-Conscious Explorers"
    else:
        return "Conservative Minimizers"

persona_summary["Persona"] = persona_summary.apply(assign_persona, axis=1)
persona_summary = persona_summary[["Persona"] + persona_features + ["Count"]]

print(persona_summary)

plt.figure(figsize=(10, 6))
plt.bar(persona_summary.index.astype(str), persona_summary["Total_Spend"], color=sns.color_palette("viridis", len(persona_summary)))
plt.title("Average Total Spend per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Average Total Spend")
plt.show()

plt.figure(figsize=(10, 6))
plt.bar(persona_summary.index.astype(str), persona_summary["Income"], color=sns.color_palette("viridis", len(persona_summary)))
plt.title("Average Income per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Average Income")
plt.show()
